In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

ROOT = Path.cwd()
DATASETS = {
    "baseline_flags": ROOT / "outputs/cleaned/songs_pre_imputation_with_missingness_flags.parquet",
    "valence_final": ROOT / "valence_final.parquet",
}
FEATURE_POOL = [
    "valence",
    "loudness",
    "danceability",
    "energy",
    "tempo",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "key",
    "mode",
    "valence_missing",
    "loudness_missing",
    "danceability_missing",
    "energy_missing",
    "tempo_missing",
    "acousticness_missing",
    "instrumentalness_missing",
    "liveness_missing",
    "speechiness_missing",
    "key_missing",
    "mode_missing",
]
MODEL_PARAMS = {
    "n_estimators": 600,
    "max_depth": 6,
    "learning_rate": 0.03,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 1.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "gamma": 0.0,
    "tree_method": "hist",
    "device": "cpu",
    "random_state": 42,
    "n_jobs": -1,
}
SAMPLE_SIZE = 400_000


In [ ]:
def run_comparison(target):
    cols = sorted(set(FEATURE_POOL + [target]))
    base_df = pd.read_parquet(DATASETS["baseline_flags"], columns=cols)
    observed_df = base_df[base_df[target].notna()].copy()

    if len(observed_df) > SAMPLE_SIZE:
        observed_df, _ = train_test_split(
            observed_df,
            train_size=SAMPLE_SIZE,
            random_state=42,
            stratify=observed_df[target].astype("int32"),
        )

    split_idx = observed_df.index.to_numpy()
    y_split = observed_df[target].astype("int32")
    train_idx, test_idx = train_test_split(
        split_idx,
        test_size=0.2,
        random_state=42,
        stratify=y_split,
    )

    results = []

    for name, path in DATASETS.items():
        df = pd.read_parquet(path, columns=cols)
        float64_columns = df.select_dtypes(include=["float64"]).columns
        df[float64_columns] = df[float64_columns].astype("float32")

        feature_columns = [column for column in FEATURE_POOL if column != target]
        X_train = df.loc[train_idx, feature_columns]
        X_test = df.loc[test_idx, feature_columns]
        y_train = df.loc[train_idx, target].astype("int32")
        y_test = df.loc[test_idx, target].astype("int32")
        valence_missing_test = df.loc[test_idx, "valence_missing"].astype(bool)

        params = dict(MODEL_PARAMS)
        if target == "mode":
            params["objective"] = "binary:logistic"
            params["eval_metric"] = "logloss"
        else:
            params["objective"] = "multi:softprob"
            params["num_class"] = 12
            params["eval_metric"] = "mlogloss"

        model = XGBClassifier(**params)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        results.append({
            "dataset": name,
            "target": target,
            "sample_rows": int(len(split_idx)),
            "test_rows": int(len(y_test)),
            "valence_missing_test_rows": int(valence_missing_test.sum()),
            "accuracy": float(accuracy_score(y_test, predictions)),
            "f1_macro": float(f1_score(y_test, predictions, average="macro")),
            "subset_accuracy": float(accuracy_score(y_test[valence_missing_test], predictions[valence_missing_test])),
            "subset_f1_macro": float(f1_score(y_test[valence_missing_test], predictions[valence_missing_test], average="macro")),
        })

    return pd.DataFrame(results)


In [ ]:
mode_results = run_comparison("mode")
key_results = run_comparison("key")
results = pd.concat([mode_results, key_results], ignore_index=True)
results.to_csv(ROOT / "valence_initial_dataset_comparison.csv", index=False)
results


In [ ]:
improvements = []

for target in ["mode", "key"]:
    base_row = results[(results["dataset"] == "baseline_flags") & (results["target"] == target)].iloc[0]
    valence_row = results[(results["dataset"] == "valence_final") & (results["target"] == target)].iloc[0]
    improvements.append({
        "target": target,
        "accuracy_gain": float(valence_row["accuracy"] - base_row["accuracy"]),
        "f1_macro_gain": float(valence_row["f1_macro"] - base_row["f1_macro"]),
        "subset_accuracy_gain": float(valence_row["subset_accuracy"] - base_row["subset_accuracy"]),
        "subset_f1_macro_gain": float(valence_row["subset_f1_macro"] - base_row["subset_f1_macro"]),
    })

pd.DataFrame(improvements)
